# Exercise on One-Hot Encoding

## Imports

In [1]:
import importlib
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import pandas as pd

In [75]:
import sys
sys.path.append('..')

import models
import toolkit

importlib.reload(models)
importlib.reload(toolkit)

from models import *
from toolkit import *

model_kit = ModelKit()
stat_kit = StatKit()

model_1 = MultiLinearRegression()
model_1_I = MultiLinearRegression()
model_1_NP = MultiLinearRegression()
model_1_NP_I = MultiLinearRegression()

model_2 = MultiLinearRegression()
model_3 = MultiLinearRegression()

## Preface 

> A better way to understand the one-hot encoding and intercept concept

Let's have 3 job titles: Engineer, Nurse, and Pilot. We encode them with [1,0,0], [0,1,0], and [0,0,1]. One has to be only one of them; meaning there is no such input as [0,0,0] or [1,1,0].

Now let's have a toy dataset of these job titles and their salaries.

In [48]:
df = pd.DataFrame({
    "Job_Title": ["Engineer", "Nurse", "Engineer", "Pilot", "Pilot", "Nurse"],
    "Salary": [81_000, 63_000, 85_500, 120_000, 150_000, 58_000]
})

df

,Job_Title,Salary
0,Engineer,81000
1,Nurse,63000
2,Engineer,85500
3,Pilot,120000
4,Pilot,150000
5,Nurse,58000


Let's apply one-hot encoding.

In [49]:
df_enc = stat_kit.one_hot_encode(df, "Job_Title")
df_enc

,Salary,Job_Title_Engineer,Job_Title_Nurse,Job_Title_Pilot
0,81000,1.0,0.0,0.0
1,63000,0.0,1.0,0.0
2,85500,1.0,0.0,0.0
3,120000,0.0,0.0,1.0
4,150000,0.0,0.0,1.0
5,58000,0.0,1.0,0.0


Now begins the main part.

First, let's just take Nurse and Pilot data (including the data points where both are zero) and train a model based on them.

In [50]:
X_NP = df_enc[["Job_Title_Nurse", "Job_Title_Pilot"]].to_numpy(dtype=np.float64)
y_NP = df_enc["Salary"].to_numpy(dtype=np.float64)

print(X_NP)
print(y_NP)

[[0. 0.]
 [1. 0.]
 [0. 0.]
 [0. 1.]
 [0. 1.]
 [1. 0.]]
[ 81000.  63000.  85500. 120000. 150000.  58000.]


Because we have two inputs and no intercept, the general formula the model predicts becomes $y=b_1x_1+b_2x_2$ meaning when both inputs are zero, the prediction salary is zero as well.

In [81]:
model_1_NP.train(X_NP, y_NP)
model_1_NP.params()

array([ 60500., 135000.])

Because the prediction salary is zero when both inputs are zero, the plane passes through the origin.

$y = 0.61x_1 + 1.35x_2$

*(parameters are scaled down by a factor of 100,000 for the sake of better visualizations)*

![Nurse, Pilot and their salaries](nurse_and_pilot.png)

Here is a key thing you need to know. In this case, Engineer salaries, i.e. rows of [0,0], don't have any impact on value of the parameters. In fact, we can remove those rows and still get the same parameters $b_1$ and $b_2$. Why? Below are explained two ways to understand this:

1. In the closed-form $(X^TX)^{-1}X^Ty$, the zeros in $X$ matrix mathematically obliterate the Engineer's salary from vector $y$ during the matrix multiplication X^Ty.

2. Looking through the lens of Sum of Squared Errors,
    
    $$\text{Error} = \sum_{i=1}^n(y_i-b_1x_{(i,1)}-b_2x_{(i,2)})^2$$
    
    isolating this formula specifically for an Engineer row (where $y=81,000$, $x_1=0$, and $x_2=0$) looks like:

    $$
    \text{Error}_\text{Engineer} = \sum_{i=1}^1(81000 - b_1(0) - b_2(0))^2\\
    \text{Error}_\text{Engineer} = (81000)^2\\
    \text{Error}_\text{Engineer} = 6,561,000,000
    $$

    Notice that $b_1$ and $b_2$ have completely disappeared from the Engineer's error.

    Optimization (finding the "best fit") works by asking a simple question: *"If I tweak $b_1$ or $b_2$ just a little bit, does my error go up or down?"* 
    
    - For the Nurse and Pilot rows, changing $b_1$ and $b_2$ actively shifts the prediction, causing their error to go up or down. The model tunes $b_1$ and $b_2$ to minimize that error.

    - But for the Engineer rows, the error is constant. However the model changes the parameters, the error for the Engineer salary doesn't change. That is why the leftover Engineer rows [salary, 0, 0] don't have any impact on the model's decision of what the parameters should be.

3. Geometrically thinking, imagine the Engineer's salary error magnitude which lies on the z-axis because $x_1=0$ and $x_2=0$. The best fit plane passes through the origin. So, however you change $b_1$ and $b_2$, meaning the tilt of the plane, the Engineer's error won't change. Meaning there is no connection between the parameters and the Engineer's salary, so we can even remove the rows and still get the same parameters.

---

---

Now let's include an intercept column and see what will happen.

In [82]:
model_1_NP_I.train(X_NP, y_NP, fit_intercept=True)
model_1_NP_I.params()

array([ 83250., -22750.,  51750.])

This can be interpreted as setting 8.33 as a base salary. The model deducts 2.28 to find the best fit salary of a Nurse, and adds 5.18 to find the Pilot's.

$y = 8.33 - 2.28x_1 + 5.18x_2$

*(parameters are scaled down by a factor of 10,000 for the sake of better visualizations)*

Carefully thinking about the situation we have at hand, when the person is neither a Nurse nor a Pilot, their salary is 8.33. ***Based on our dataset's definition, a person that is neither a Nurse nor a Pilot is an Engineer. So this intercept value 8.33 can be interpreted as the best fit salary of an Engineer.***

![Nurse, Pilot and their salaries including intercept](intercept_nurse_and_pilot.png)

## Exercise 1: Single Categorical Feature

### Setup

- **Goal**: Predict Price based solely on the type of Fruit.
- **Task**: 
    - One-hot encode 'Fruit',
    - drop one category,
    - train a model, and
    - interpret the intercept.

In [ ]:
df1 = pd.DataFrame({
    "Fruit": ["Apple", "Apple", "Banana", "Banana", "Cherry", "Cherry"],
    "Price": [1.2, 1.3, 0.5, 0.6, 3.0, 3.2]
})
df1

,Fruit,Price
0,Apple,1.2
1,Apple,1.3
2,Banana,0.5
3,Banana,0.6
4,Cherry,3.0
5,Cherry,3.2


### Data preparation

In [39]:
df1_enc = stat_kit.one_hot_encode(df1, columns="Fruit")

In [40]:
df1_enc

,Price,Fruit_Apple,Fruit_Banana,Fruit_Cherry
0,1.2,1.0,0.0,0.0
1,1.3,1.0,0.0,0.0
2,0.5,0.0,1.0,0.0
3,0.6,0.0,1.0,0.0
4,3.0,0.0,0.0,1.0
5,3.2,0.0,0.0,1.0


In [87]:
X1 = df1_enc[["Fruit_Apple", "Fruit_Banana", "Fruit_Cherry"]].to_numpy(dtype=np.float64)
X1_I = df1_enc[["Fruit_Banana", "Fruit_Cherry"]].to_numpy(dtype=np.float64)
y1 = df1_enc["Price"].to_numpy(dtype=np.float64)

In [88]:
print(X1)
print(y1)

[[1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 1.]]
[1.2 1.3 0.5 0.6 3.  3.2]


### Model training

In [89]:
model_1.train(features=X1, label=y1)

print(model_1.params())

[1.25 0.55 3.1 ]


By training all the features with no intercept, we get the best fit equation:

$$y = 1.25x_1 + 0.55x_2 + 3.1x_3$$

---

In [90]:
model_1_I.train(features=X1_I, label=y1, fit_intercept=True)

print(model_1_I.params())

[ 1.25 -0.7   1.85]


By dropping the banana feature and including an intercept column, training gives us:
$$y = 1.25 - 0.7x_1 + 1.85x_2$$

The model doesn't have a clue what those two equations represent. It is our responsibility to interpret them in a meaningful way.

### Conclusion

**WITHOUT** the intercept, the model solved for the following parameters:

- general formula: $y = b_1x_1 + b_2x_2 + b_3x_3$ 
- weight of **Apple**: $b_1 = 1.25$
- weight of **Banana**: $b_2 = 0.55$
- weight of **Cherry**: $b_3 = 3.1$ 
- best fit formula: $\boxed{y = 1.25x_1 + 0.55x_2 + 3.1x_3}$

The model's predictions are:
- Price of Apple [1, 0, 0] = 1.25.
- Price of Banana [0, 1, 0] = 0.55.
- Price of Cherry [0, 0, 1] = 3.1.

**WITH** the intercept, the model solved for the following parameters:
- general formula $y = b_0 + b_1x_1 + b_2x_2$
- the **Intercept** term $b_0 = 1.25$
- weight of **Banana** $b_1 = -0.7$
- weight of **Cherry** $b_2 = 1.85$
- general formula $\boxed{y = 1.25 - 0.7x_1 + 1.85x_2}$

The model sets the price of apple as a base price. It doesn't intentionally set it though, it's just the way OLS works. The intercept term that minimizes the sum of squared errors becomes apple's price because of the nature of the linear dependency among the features.

We know that apple is given when the encoded values for banana and cherry are both 0, meaning when all the inputs are zero, we get apple's price (technical definition of y-intercept). However, the inputs (apple, banana and cherry) can't all be 0 at the same time; one has to be given. That's why we are so certain that if banana and cherry are zero, it means the input is apple.

With all the features together (including the intercept column), the X matrix looks like this:
$$X = \begin{bmatrix}
1 & 1 & 0 & 0\\
1 & 0 & 1 & 0\\
1 & 1 & 0 & 0\\
1 & 0 & 0 & 1\\
1 & 0 & 0 & 1\\
1 & 0 & 1 & 0\\
\vdots & \vdots & \vdots & \vdots
\end{bmatrix}
$$

The meaning of linear dependency by itself means that one vector (feature) can be expressed by the span of the others. Meaning, if we remove that vector (feature), we don't lose critical data as it can be regained by the addition and scaling of other features.

## Exercise 2

In [ ]:
# Exercise 2: Categorical + Continuous Feature
# Goal: Predict Salary based on Degree type and Years of Experience.
# Task: One-hot encode 'Degree'. Notice how the interpretation of the intercept changes now that 'Experience' is in the equation.
df2 = pd.DataFrame({
    "Degree": ["Bachelors", "Bachelors", "Masters", "Masters", "PhD", "PhD"],
    "Experience": [1, 3, 2, 5, 2, 6],
    "Salary": [65, 75, 80, 95, 95, 115] 
})
df2

## Exercise 3

In [ ]:
# Exercise 3: Multiple Categorical Features
# Goal: Predict Rent based on City and Property Type.
# Task: One-hot encode BOTH 'City' and 'Type'. Drop one category from each. 
# Observe what combination of features the intercept (grand baseline) represents.
df3 = pd.DataFrame({
    "City": ["London", "London", "Paris", "Paris", "NY", "NY"],
    "Type": ["Apartment", "House", "Apartment", "House", "Apartment", "House"],
    "Rent": [2000, 3000, 1800, 2800, 2500, 3500]
})
df3